# Feature Selection Lab - Breast Cancer Dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

print("Shape:", X.shape)
X.head()

Shape: (569, 30)


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [2]:
# check missing values, duplicates and class counts
print("Missing:", X.isnull().sum().sum())
print("Duplicates:", X.duplicated().sum())
print(y.value_counts())

Missing: 0
Duplicates: 0
target
1    357
0    212
Name: count, dtype: int64


In [3]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

# scale features manually
scaler = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_sc = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Part A - Baseline (All 30 features)

In [4]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

results_base = []

# 1. Dummy
t0 = time.time()
dummy = DummyClassifier(strategy='most_frequent')
cv_acc = cross_val_score(dummy, X_train_sc, y_train, cv=cv, scoring='accuracy').mean()
cv_f1 = cross_val_score(dummy, X_train_sc, y_train, cv=cv, scoring='f1').mean()
dummy.fit(X_train_sc, y_train)
pred = dummy.predict(X_test_sc)
results_base.append(['Dummy', cv_acc, cv_f1, accuracy_score(y_test, pred), f1_score(y_test, pred), time.time() - t0])

# 2. Logistic Regression
t0 = time.time()
lr = LogisticRegression(max_iter=1000, random_state=42)
cv_acc = cross_val_score(lr, X_train_sc, y_train, cv=cv, scoring='accuracy').mean()
cv_f1 = cross_val_score(lr, X_train_sc, y_train, cv=cv, scoring='f1').mean()
lr.fit(X_train_sc, y_train)
pred = lr.predict(X_test_sc)
results_base.append(['Logistic Regression', cv_acc, cv_f1, accuracy_score(y_test, pred), f1_score(y_test, pred), time.time() - t0])

# 3. SVC
t0 = time.time()
svc = SVC(random_state=42)
cv_acc = cross_val_score(svc, X_train_sc, y_train, cv=cv, scoring='accuracy').mean()
cv_f1 = cross_val_score(svc, X_train_sc, y_train, cv=cv, scoring='f1').mean()
svc.fit(X_train_sc, y_train)
pred = svc.predict(X_test_sc)
results_base.append(['SVC', cv_acc, cv_f1, accuracy_score(y_test, pred), f1_score(y_test, pred), time.time() - t0])

# 4. Random Forest
t0 = time.time()
rf = RandomForestClassifier(random_state=42)
cv_acc = cross_val_score(rf, X_train, y_train, cv=cv, scoring='accuracy').mean()
cv_f1 = cross_val_score(rf, X_train, y_train, cv=cv, scoring='f1').mean()
rf.fit(X_train, y_train)
pred = rf.predict(X_test)
results_base.append(['Random Forest', cv_acc, cv_f1, accuracy_score(y_test, pred), f1_score(y_test, pred), time.time() - t0])

df_base = pd.DataFrame(results_base, columns=['Model', 'CV_Acc', 'CV_F1', 'Test_Acc', 'Test_F1', 'Time_sec'])
df_base.round(4)

,Model,CV_Acc,CV_F1,Test_Acc,Test_F1,Time_sec
0,Dummy,0.6282,0.7716,0.6257,0.7698,0.5784
1,Logistic Regression,0.9824,0.9863,0.9883,0.9907,1.3125
2,SVC,0.9723,0.9781,0.9766,0.9813,0.9716
3,Random Forest,0.9623,0.9698,0.9357,0.9488,25.2533


## Part B - Filter Methods

In [5]:
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif, mutual_info_classif

# Variance Threshold
vt = VarianceThreshold(threshold=0.5)
vt.fit(X_train_sc)
vt_cols = X_train_sc.columns[vt.get_support()]
print("Features kept after VarianceThreshold:", len(vt_cols))

# ANOVA F-test (top 10)
skb_f = SelectKBest(score_func=f_classif, k=10)
skb_f.fit(X_train_sc, y_train)
f_cols = X_train_sc.columns[skb_f.get_support()]
print("Top 10 ANOVA features:", list(f_cols))

# Mutual Information (top 10)
skb_mi = SelectKBest(score_func=mutual_info_classif, k=10)
skb_mi.fit(X_train_sc, y_train)
mi_cols = X_train_sc.columns[skb_mi.get_support()]
print("Top 10 MI features:", list(mi_cols))

Features kept after VarianceThreshold: 30
Top 10 ANOVA features: ['mean radius', 'mean perimeter', 'mean area', 'mean concavity', 'mean concave points', 'worst radius', 'worst perimeter', 'worst area', 'worst concavity', 'worst concave points']
Top 10 MI features: ['mean radius', 'mean perimeter', 'mean area', 'mean concavity', 'mean concave points', 'worst radius', 'worst perimeter', 'worst area', 'worst concavity', 'worst concave points']


In [6]:
# Evaluate Logistic Regression and SVC on ANOVA top 10
lr_f = LogisticRegression(max_iter=1000, random_state=42)
lr_f.fit(X_train_sc[f_cols], y_train)
print("LR (ANOVA 10) Test Acc:", round(accuracy_score(y_test, lr_f.predict(X_test_sc[f_cols])), 4))

svc_f = SVC(random_state=42)
svc_f.fit(X_train_sc[f_cols], y_train)
print("SVC (ANOVA 10) Test Acc:", round(accuracy_score(y_test, svc_f.predict(X_test_sc[f_cols])), 4))

# Evaluate on Mutual Info top 10
lr_mi = LogisticRegression(max_iter=1000, random_state=42)
lr_mi.fit(X_train_sc[mi_cols], y_train)
print("LR (MI 10) Test Acc:", round(accuracy_score(y_test, lr_mi.predict(X_test_sc[mi_cols])), 4))

svc_mi = SVC(random_state=42)
svc_mi.fit(X_train_sc[mi_cols], y_train)
print("SVC (MI 10) Test Acc:", round(accuracy_score(y_test, svc_mi.predict(X_test_sc[mi_cols])), 4))

LR (ANOVA 10) Test Acc: 0.9357
SVC (ANOVA 10) Test Acc: 0.9415
LR (MI 10) Test Acc: 0.9357
SVC (MI 10) Test Acc: 0.9415


## Part C - Wrapper Methods (SFS & RFE)

In [ ]:
from sklearn.feature_selection import SequentialFeatureSelector, RFE, RFECV

# Forward Sequential Feature Selection with Logistic Regression
t0 = time.time()
sfs = SequentialFeatureSelector(LogisticRegression(max_iter=1000, random_state=42), n_features_to_select=10, direction='forward', cv=cv)
sfs.fit(X_train_sc, y_train)
sfs_cols = X_train_sc.columns[sfs.get_support()]
print("SFS took:", round(time.time() - t0, 2), "sec")
print("SFS selected:", list(sfs_cols))

lr_sfs = LogisticRegression(max_iter=1000, random_state=42)
lr_sfs.fit(X_train_sc[sfs_cols], y_train)
print("LR (SFS 10) Test Acc:", round(accuracy_score(y_test, lr_sfs.predict(X_test_sc[sfs_cols])), 4))

In [ ]:
# RFE with Logistic Regression
rfe = RFE(estimator=LogisticRegression(max_iter=1000, random_state=42), n_features_to_select=10)
rfe.fit(X_train_sc, y_train)
rfe_cols = X_train_sc.columns[rfe.get_support()]
print("RFE selected:", list(rfe_cols))

lr_rfe = LogisticRegression(max_iter=1000, random_state=42)
lr_rfe.fit(X_train_sc[rfe_cols], y_train)
print("LR (RFE 10) Test Acc:", round(accuracy_score(y_test, lr_rfe.predict(X_test_sc[rfe_cols])), 4))

# RFECV to find optimal number of features automatically
rfecv = RFECV(estimator=LogisticRegression(max_iter=1000, random_state=42), step=1, cv=cv, scoring='accuracy')
rfecv.fit(X_train_sc, y_train)
print("Optimal features by RFECV:", rfecv.n_features_)
rfecv_cols = X_train_sc.columns[rfecv.get_support()]

lr_rfecv = LogisticRegression(max_iter=1000, random_state=42)
lr_rfecv.fit(X_train_sc[rfecv_cols], y_train)
print("LR (RFECV) Test Acc:", round(accuracy_score(y_test, lr_rfecv.predict(X_test_sc[rfecv_cols])), 4))

RFE selected: ['mean area', 'mean concave points', 'area error', 'worst radius', 'worst texture', 'worst perimeter', 'worst area', 'worst smoothness', 'worst concave points', 'worst symmetry']
LR (RFE 10) Test Acc: 0.9708
Optimal features by RFECV: 8
LR (RFECV) Test Acc: 0.9591
